# Práctica 3 · Medir el rendimiento

**10–15 minutos · Continuación de la práctica 1**

Ya sabemos generar texto. Ahora mediremos cuánto tarda y cómo llegan los tokens.
Usaremos **Qwen2.5-0.5B-Instruct**, más pequeño que el modelo de la primera práctica.

Abre este cuaderno en un entorno nuevo de Colab y ejecuta las celdas en orden.
GPU recomendada; también funciona en CPU, más despacio. La descarga y la carga del modelo quedan fuera de las mediciones.

## 1. Preparar y cargar
En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. PyTorch ya viene instalado.

In [ ]:
%pip -q install "transformers==4.57.1" pandas matplotlib

In [ ]:
%matplotlib inline

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from transformers.generation.streamers import BaseStreamer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, torch_dtype=dtype, use_safetensors=True,
).to(device).eval()
# Generación greedy: misma configuración en todas las pruebas.
config = GenerationConfig.from_model_config(model.config)
config.eos_token_id = model.generation_config.eos_token_id
config.pad_token_id = tokenizer.pad_token_id
hardware = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"{MODEL_ID} · {hardware} · {dtype}")
print(f"PyTorch {torch.__version__} · Transformers {transformers.__version__}")

## 2. Poner un reloj a los tokens

| Métrica | Qué medimos aquí |
|---|---|
| TTFT | Desde que empieza la función hasta recibir el primer token. |
| ITL | Tiempo entre dos tokens consecutivos. |
| TPOT | Media de esos intervalos, después del primer token. |
| Latencia E2E | Preparar el prompt + generar + decodificar la respuesta. |
| Tokens/s | Tokens de salida divididos entre la latencia E2E. |

El reloj recibe **IDs de tokens**, no palabras. El streamer añade algo de trabajo: estas son medidas locales con instrumentación, sin red ni API. Contamos también un posible token de fin de secuencia.

In [ ]:
class RelojTokens(BaseStreamer):
    def __init__(self):
        self.es_prompt = True
        self.tiempos = []

    def put(self, tokens):
        if self.es_prompt:  # generate() entrega primero el prompt completo.
            self.es_prompt = False
            return
        assert tokens.numel() == 1, "Esta práctica mide una secuencia cada vez."
        self.tiempos.append(time.perf_counter())

    def end(self):
        pass


def medir(prompt, max_new_tokens=32):
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    inicio = time.perf_counter()
    texto = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True,
    )
    entradas = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(device)
    reloj = RelojTokens()
    with torch.inference_mode():
        salida = model.generate(
            **entradas, generation_config=config, do_sample=False,
            max_new_tokens=max_new_tokens, use_cache=True, streamer=reloj,
        )
    if device.type == "cuda":
        torch.cuda.synchronize()
    n_entrada = entradas["input_ids"].shape[-1]
    n_salida = salida.shape[-1] - n_entrada
    respuesta = tokenizer.decode(salida[0, n_entrada:].cpu(), skip_special_tokens=True)
    e2e = time.perf_counter() - inicio
    assert len(reloj.tiempos) == n_salida > 0
    itl = np.diff(reloj.tiempos) * 1000
    return {
        "tokens_entrada": n_entrada, "tokens_salida": n_salida,
        "prompt_KiB": len(texto.encode("utf-8")) / 1024,
        "ttft_ms": (reloj.tiempos[0] - inicio) * 1000,
        "itl_p50_ms": np.median(itl) if len(itl) else np.nan,
        "itl_p95_ms": np.percentile(itl, 95) if len(itl) else np.nan,
        "tpot_ms": np.mean(itl) if len(itl) else np.nan,
        "e2e_ms": e2e * 1000, "salida_tokens_s": n_salida / e2e,
        "gpu_pico_asignado_MiB": torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else np.nan,
        "gpu_pico_reservado_MiB": torch.cuda.max_memory_reserved() / 2**20 if device.type == "cuda" else np.nan,
        "llegada_ms": [(t - inicio) * 1000 for t in reloj.tiempos],
        "respuesta": respuesta,
    }

## 3. Comparar dos prompts
Solo cambia la entrada. Mantenemos modelo, precisión, generación greedy y límite de salida.
La respuesta puede terminar antes del límite: por eso contamos los tokens reales.

In [ ]:
PROMPTS = {
    "Corto": "Explica en español qué es una API, en dos frases.",
    "Largo": (
        "Lee este contexto: "
        + "Una API permite que dos programas intercambien datos mediante peticiones y respuestas. " * 12
        + "Explica en español qué es una API, en dos frases."
    ),
}
REPETICIONES = 3
MAX_NUEVOS_TOKENS = 32

# Calentamiento de ambos casos: no entra en la tabla ni en las tasas.
for prompt in PROMPTS.values():
    medir(prompt, max_new_tokens=8)
print("Modelo preparado. Ahora sí medimos.")

In [ ]:
filas = []
inicio_lote = time.perf_counter()
for caso, prompt in PROMPTS.items():
    for repeticion in range(1, REPETICIONES + 1):
        try:
            filas.append({"caso": caso, "repeticion": repeticion, "estado": "OK", **medir(prompt, MAX_NUEVOS_TOKENS)})
        except Exception as error:
            filas.append({"caso": caso, "repeticion": repeticion, "estado": "ERROR", "error": str(error)})
segundos_lote = time.perf_counter() - inicio_lote
resultados = pd.DataFrame(filas)
correctas = resultados[resultados["estado"] == "OK"].copy()
if correctas.empty:
    display(resultados)
    raise RuntimeError("No hay generaciones correctas. Revisa los errores antes de continuar.")

columnas = ["caso", "repeticion", "tokens_entrada", "tokens_salida", "prompt_KiB",
            "ttft_ms", "itl_p50_ms", "itl_p95_ms", "tpot_ms", "e2e_ms", "salida_tokens_s"]
display(correctas[columnas].round(2))
print("Ejemplo de respuesta:\n", correctas.iloc[0]["respuesta"])

## 4. Ver los resultados
El primer punto de cada curva muestra el TTFT. Después, la pendiente refleja el ritmo de generación.
Las barras resumen las repeticiones con la mediana.

In [ ]:
resumen = correctas.groupby("caso", sort=False).agg(
    ttft_ms=("ttft_ms", "median"), e2e_ms=("e2e_ms", "median"),
    salida_tokens_s=("salida_tokens_s", "median"),
    tpot_ms=("tpot_ms", "median"),
    e2e_p95_ms=("e2e_ms", lambda valores: valores.quantile(.95)),
)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for caso, grupo in correctas.groupby("caso", sort=False):
    fila = grupo.iloc[0]
    axes[0].plot(np.array(fila["llegada_ms"]) / 1000,
                 np.arange(1, fila["tokens_salida"] + 1), label=caso)
axes[0].set(xlabel="Segundos desde el inicio", ylabel="Tokens recibidos", title="Llegada de tokens · una ejecución")
axes[0].legend()
resumen[["ttft_ms", "e2e_ms"]].plot.bar(ax=axes[1], color=["#5368bd", "#aab6cd"], rot=0)
axes[1].set(xlabel="Prompt", ylabel="Milisegundos", title="Latencia mediana")
axes[1].legend(["Primer token", "Respuesta completa"])
resumen["salida_tokens_s"].plot.bar(ax=axes[2], color="#5368bd", rot=0)
axes[2].set(xlabel="Prompt", ylabel="Tokens/s · tiempo E2E", title="Velocidad mediana")
fig.tight_layout()
plt.show()
display(resumen.round(2))

In [ ]:
completadas = len(correctas)
errores = len(resultados) - completadas
print(f"Generaciones correctas: {completadas}/{len(resultados)}")
print(f"Errores: {errores / len(resultados):.1%}")
print(f"Generaciones correctas/min, secuenciales: {60 * completadas / segundos_lote:.2f}")
print(f"Salida total, secuencial: {correctas['tokens_salida'].sum() / segundos_lote:.2f} tokens/s")
if errores:
    display(resultados[resultados["estado"] == "ERROR"][["caso", "repeticion", "error"]])
if device.type == "cuda":
    display(correctas[["caso", "repeticion", "gpu_pico_asignado_MiB", "gpu_pico_reservado_MiB"]].round(1))
else:
    print("Memoria CUDA: no disponible en CPU; no se sustituye por un cero.")

**Cómo leerlo**
- Son generaciones locales **secuenciales**, no peticiones a un servidor ni su capacidad bajo concurrencia.
- Tres repeticiones por caso sirven para practicar. Su p95 es muestral: no valida un SLO ni una tasa de errores de producción.
- `prompt_KiB` mide texto UTF-8 con plantilla de chat; no mide KV cache. La memoria CUDA es la del asignador de PyTorch e incluye los pesos; no equivale a toda la VRAM observada en `nvidia-smi`.
- Cada llamada empieza sin KV previa. `use_cache=True` reutiliza KV dentro de esa generación; no es prefix caching entre peticiones. Cache hit, cola y utilización de GPU requieren instrumentación adicional del motor o del servidor.

## 5. Tu prueba
Cambia **solo** `MAX_NUEVOS_TOKENS` a 64 y repite desde la celda de medición. ¿Qué cambia más: TTFT o latencia total?
Después puedes alargar el contexto. Para contrastar resultados, usa la misma GPU, precisión, prompts y configuración.

In [ ]:
# En Colab, descarga metricas.csv desde el panel Archivos.
exportacion = resultados.drop(columns=["llegada_ms", "respuesta"], errors="ignore").assign(
    modelo=MODEL_ID, revision=REVISION, hardware=hardware, precision=str(dtype),
    max_nuevos_tokens=MAX_NUEVOS_TOKENS,
    version_torch=torch.__version__, version_transformers=transformers.__version__,
)
exportacion.to_csv("metricas.csv", index=False)
print("Guardado: metricas.csv")

### Referencias
- [Modelo Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
- [Streaming de generación en Transformers](https://huggingface.co/docs/transformers/v4.57.1/en/generation_features)
- [Memoria CUDA en PyTorch](https://docs.pytorch.org/docs/stable/notes/cuda.html#cuda-memory-management)